In [2]:
import sklearn, lightgbm, xgboost, catboost, optuna, shap
print("sklearn", sklearn.__version__)    # 1.0 이상 (StratifiedGroupKFold 필요)
print("lightgbm", lightgbm.__version__)
print("xgboost", xgboost.__version__)    # 2.0 이상 가정 (early_stopping_rounds 생성자 방식)
print("catboost", catboost.__version__)
print("optuna", optuna.__version__)
print("shap", shap.__version__)

sklearn 1.8.0
lightgbm 4.6.0
xgboost 3.2.0
catboost 1.2.10
optuna 4.8.0
shap 0.51.0


In [3]:
# ==============================================================
# 라이브러리 + 한글 폰트
# ==============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform, os, json, warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, accuracy_score,
                             classification_report, confusion_matrix)
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
import shap
import joblib

# ★ 순서 중요: sns.set_style이 font.family를 덮어쓰므로 반드시 폰트 설정보다 먼저!
sns.set_style("whitegrid")
def set_korean_font():
    s = platform.system()
    if s == "Darwin":      plt.rcParams["font.family"] = "AppleGothic"
    elif s == "Windows":   plt.rcParams["font.family"] = "Malgun Gothic"
    else:                  plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
set_korean_font()

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 100)
os.makedirs("../../figures", exist_ok=True)
os.makedirs("../../models", exist_ok=True)
os.makedirs("../../data/processed/4_model", exist_ok=True)
SEED = 42

In [4]:
# ==============================================================
# step11 로드 + 분할(split_assignment) 병합
# ==============================================================
df = pd.read_csv("../../data/processed/3_eda/step11_features.csv",
                 encoding="utf-8-sig", low_memory=False)
print(f"데이터: {df.shape}")   # (2408699, 92)

SPLIT_PATH = "../../data/processed/4_model/split_assignment.csv"

if os.path.exists(SPLIT_PATH):
    split = pd.read_csv(SPLIT_PATH, encoding="utf-8-sig")
    print("기존 분할 로드 (회귀팀과 공유) — 새로 만들지 말 것!")
else:
    # 최초 1회만 실행됨 — 농장 단위 + 등급 비율 유지 분할
    from sklearn.model_selection import StratifiedGroupKFold
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    dev_idx, hold_idx = next(sgkf.split(df, df["LAST_GRADE"],
                                        groups=df["FARM_UNIQUE_NO"]))
    df["split"] = "dev"; df.loc[hold_idx, "split"] = "holdout"
    df["fold"] = -1
    dev_tmp = df[df["split"] == "dev"]
    sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for k, (_, va) in enumerate(sgkf2.split(dev_tmp, dev_tmp["LAST_GRADE"],
                                            groups=dev_tmp["FARM_UNIQUE_NO"])):
        df.loc[dev_tmp.index[va], "fold"] = k
    df[["CATTLE_NO","split","fold"]].to_csv(SPLIT_PATH, index=False,
                                            encoding="utf-8-sig")
    split = df[["CATTLE_NO","split","fold"]]
    print("분할 새로 생성 후 저장")

df = df.merge(split, on="CATTLE_NO", how="left") if "split" not in df.columns else df
print(df["split"].value_counts())
print("dev fold 분포:\n", df[df['split']=='dev']['fold'].value_counts().sort_index())

# 분할 자체 검증 — 농장이 두 영역에 걸치면 누수
farm_check = df.groupby("FARM_UNIQUE_NO")["split"].nunique()
assert (farm_check > 1).sum() == 0, "농장 누수 발견! 분할 파일 확인 필요"
print("농장 누수 검증 통과")

데이터: (2408699, 92)
기존 분할 로드 (회귀팀과 공유) — 새로 만들지 말 것!
split
dev        1926959
holdout     481740
Name: count, dtype: int64
dev fold 분포:
 fold
0    385392
1    385392
2    385392
3    385391
4    385392
Name: count, dtype: int64
농장 누수 검증 통과


In [6]:
# ==============================================================
# 트랙1 피처 — "test에서 계산 가능한가"로 선별
# ==============================================================
EXCLUDE = {
    # ① 타깃 자체·타깃 파생
    "LAST_GRADE", "grade_num",
    # ② test에 없는 도축 후 측정값 (정답 누수 — 절대 금지)
    "BACKFAT","REA","WINDEX","INSFAT","YUKSAK","FATSAK","TISSUE","GROWTH",
    "WGRADE","COST_AMT",
    # ③ ID·해시 (그 자체로는 의미 없음 — 빈도 인코딩본을 대신 사용)
    "CATTLE_NO","FARM_UNIQUE_NO","stn",
    "KPN_NO","FATHER_CATTLE_NO","MOTHER_ANIMAL_NO",
    "F_GMOTHER_ANIMAL_NO","F_GFATHER_CATTLE_NO",
    "M_GMOTHER_ANIMAL_NO","M_GFATHER_CATTLE_NO",
    # ④ 날짜 원본 (연/월/분기/계절 파생으로 대체)
    "ABATT_DATE","JUDGE_DATE","BIRTH_YMD",
    # ⑤ 문자 범주 원본 (원-핫 더미로 대체)
    "sido","sigungu","eupmyeondong","JUDGE_SEX",
    "abatt_season","birth_season","farm_size","고온_bin",
    # ⑥ 상수 (THI 폐사 등급은 데이터 기간에 0일)
    "days_폐사",
    # ⑦ 분할 정보
    "split","fold",
}
features_track1 = [c for c in df.columns
                   if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])]
print(f"트랙1 피처: {len(features_track1)}개")   # 58개

# 참고-사후 비교용 (육질 포함 — 제출 불가, 보고서 비교 전용)
QUALITY = ["BACKFAT","REA","WINDEX","INSFAT","YUKSAK","FATSAK","TISSUE","GROWTH"]
features_post = features_track1 + QUALITY
print(f"참고-사후 피처: {len(features_post)}개")  # 66개

트랙1 피처: 59개
참고-사후 피처: 67개


In [7]:
# ==============================================================
# 등급 → 정수 (16개, 품질 순서대로)
# ==============================================================
GRADE_ORDER = ["등외","3C","3B","3A","2C","2B","2A","1C","1B","1A",
               "1+C","1+B","1+A","1++C","1++B","1++A"]   # 나쁨 → 좋음
grade_to_idx = {g: i for i, g in enumerate(GRADE_ORDER)}
idx_to_grade = {i: g for g, i in grade_to_idx.items()}

df["y"] = df["LAST_GRADE"].map(grade_to_idx)
assert df["y"].isnull().sum() == 0, "매핑 안 된 등급 존재!"
n_classes = 16
print(df["LAST_GRADE"].value_counts().reindex(GRADE_ORDER).to_string())

LAST_GRADE
등외        5540
3C       23542
3B       97576
3A       47214
2C       62938
2B      195866
2A      127410
1C      116976
1B      299290
1A      168174
1+C     130094
1+B     311222
1+A     167246
1++C    128997
1++B    319588
1++A    207026


In [8]:
# ==============================================================
# dev/holdout 분리 + 결측 처리 유틸
# ==============================================================
dev  = df[df["split"] == "dev"].reset_index(drop=True)
hold = df[df["split"] == "holdout"].reset_index(drop=True)
print(f"개발: {len(dev):,} / 홀드아웃: {len(hold):,}")

def prepare_X(train_X, *other_Xs):
    """train 중앙값으로 결측 대치 (누수 방지: 중앙값은 train에서만 계산).
    ① inf→NaN ② train 중앙값 대치 ③ 그래도 NaN인(전부결측) 컬럼은 제거."""
    train_X = train_X.replace([np.inf, -np.inf], np.nan)
    med = train_X.median(numeric_only=True)
    out = [train_X.fillna(med)]
    for X_ in other_Xs:
        out.append(X_.replace([np.inf, -np.inf], np.nan).fillna(med))
    # 전부 결측이라 중앙값조차 없는 컬럼 방어 (실데이터 스모크에서 실제 발생)
    bad = out[0].columns[out[0].isnull().any()].tolist()
    if bad:
        print(f"  [방어] 전부 결측 컬럼 제거: {bad}")
        out = [o.drop(columns=bad) for o in out]
    return out if len(out) > 1 else out[0]

# fold 인덱스 리스트 (위치 기반)
fold_indices = [(np.where(dev["fold"] != k)[0], np.where(dev["fold"] == k)[0])
                for k in range(5)]
y_dev = dev["y"]

# ★ heat_high 임계값을 dev 분위 하나로 통일 (train·test 동일 기준선)
#   step11의 heat_high는 전체(2.4M) 분위로 만들어졌음 → dev 분위로 다시 계산해 일관성 확보
HEAT_Q75 = float(np.nanquantile(dev["ratio_고온"], 0.75))   # ← test에서도 이 값을 재사용
for part in (dev, hold):
    part["heat_high"] = (part["ratio_고온"] >= HEAT_Q75).astype(int)
X_t1 = dev[features_track1]                                  # heat_high 갱신 반영
print(f"heat_high 임계값(dev 75분위) = {HEAT_Q75:.4f}")

개발: 1,926,959 / 홀드아웃: 481,740
heat_high 임계값(dev 75분위) = 0.4486


In [9]:
# ==============================================================
# 공통 5-fold CV 러너 — 4개 모델을 같은 조건에서
# ==============================================================
def run_cv(name, make_model, X_all, y_all, fold_indices,
           n_folds=5, needs_scaling=False, use_sample_weight=False,
           fit_kwargs_fn=None):
    """make_model(): 새 모델 객체 반환. fit_kwargs_fn(X_va,y_va): fit 추가 인자."""
    results, oof = [], np.full(len(X_all), -1, dtype=int)
    last_model = None
    for fold_i, (tr_idx, va_idx) in enumerate(fold_indices[:n_folds]):
        X_tr, X_va = X_all.iloc[tr_idx], X_all.iloc[va_idx]
        y_tr, y_va = y_all.iloc[tr_idx], y_all.iloc[va_idx]
        X_tr, X_va = prepare_X(X_tr, X_va)

        if needs_scaling:
            sc = StandardScaler()
            X_tr = pd.DataFrame(sc.fit_transform(X_tr), columns=X_tr.columns)
            X_va = pd.DataFrame(sc.transform(X_va),  columns=X_va.columns)

        model = make_model()
        kwargs = fit_kwargs_fn(X_va, y_va) if fit_kwargs_fn else {}
        if use_sample_weight:   # XGBoost용 — 클래스 균형 가중치
            kwargs["sample_weight"] = compute_sample_weight("balanced", y_tr)
        model.fit(X_tr, y_tr, **kwargs)

        pred = np.asarray(model.predict(X_va)).ravel().astype(int)  # ★CatBoost (n,1) 대응
        oof[va_idx] = pred
        f1m = f1_score(y_va, pred, average="macro")
        acc = accuracy_score(y_va, pred)
        results.append({"fold": fold_i, "macro_f1": round(f1m,4), "acc": round(acc,4)})
        print(f"  [{name}] fold {fold_i}: Macro-F1={f1m:.4f}, Acc={acc:.4f}")
        last_model = model
    r = pd.DataFrame(results)
    print(f"  [{name}] 평균 Macro-F1 = {r['macro_f1'].mean():.4f} ± {r['macro_f1'].std():.4f}")
    return r, oof, last_model

In [ ]:
# ==============================================================
# 로지스틱 회귀 — 속도 때문에 층화 서브샘플로 학습 (베이스라인 용도)
# ==============================================================
# 192만 행 × 58변수 × 16클래스 전체 학습은 매우 오래 걸림.
# 베이스라인은 "비교 기준점"이므로 30만 행 층화 샘플이면 충분.
LOGREG_N = 300_000     # TODO: 시간 여유 있으면 늘려볼 것

# X_t1 = dev[features_track1] 는 셀 6에서 이미 정의됨 (heat_high 갱신 반영)
# ★ 층화 샘플: groupby.sample(frac=)이 pandas 2.x 권장 (옛 apply+sample은 deprecation 경고)
frac = min(1.0, LOGREG_N / len(dev))
sub = dev.groupby("y", group_keys=False).sample(frac=frac, random_state=SEED).reset_index(drop=True)
sub_folds = [(np.where(sub["fold"] != k)[0], np.where(sub["fold"] == k)[0])
             for k in range(5)]
print(f"로지스틱용 샘플: {len(sub):,}행 (등급 비율 유지)")

logreg_df, logreg_oof, _ = run_cv(
    "Logistic",
    lambda: LogisticRegression(max_iter=2000, class_weight="balanced",
                               random_state=SEED),
    sub[features_track1], sub["y"], sub_folds,
    needs_scaling=True)

In [10]:
# ==============================================================
# XGBoost — 시간 절약을 위해 기본은 fold 0만 (FULL_XGB로 전환 가능)
# ==============================================================
FULL_XGB = True        # TODO: 최종 비교 때 True로 (5-fold 전체)

xgb_df, xgb_oof, xgb_last = run_cv(
    "XGBoost",
    lambda: XGBClassifier(objective="multi:softmax", num_class=n_classes,
                          learning_rate=0.05, n_estimators=500,
                          max_depth=8, min_child_weight=50,
                          subsample=0.8, colsample_bytree=0.8,
                          tree_method="hist", eval_metric="mlogloss",
                          early_stopping_rounds=50,     # ★ 2.x: 생성자에!
                          random_state=SEED, verbosity=0, n_jobs=-1),
    X_t1, y_dev, fold_indices,
    n_folds=(5 if FULL_XGB else 1),
    use_sample_weight=True,     # XGB는 class_weight가 없어 sample_weight로
    fit_kwargs_fn=lambda X_va, y_va: dict(eval_set=[(X_va, y_va)], verbose=False))

  [XGBoost] fold 0: Macro-F1=0.2024, Acc=0.1726
  [XGBoost] fold 1: Macro-F1=0.2048, Acc=0.1728
  [XGBoost] fold 2: Macro-F1=0.1995, Acc=0.1702
  [XGBoost] fold 3: Macro-F1=0.2022, Acc=0.1710
  [XGBoost] fold 4: Macro-F1=0.1990, Acc=0.1710
  [XGBoost] 평균 Macro-F1 = 0.2016 ± 0.0024


In [11]:
# ==============================================================
# CatBoost — 기본은 fold 0만 (느린 편)
# ==============================================================
FULL_CAT = True        # TODO: 최종 비교 때 True

cat_df, cat_oof, cat_last = run_cv(
    "CatBoost",
    lambda: CatBoostClassifier(loss_function="MultiClass", iterations=500,
                               learning_rate=0.05, depth=8,
                               auto_class_weights="Balanced",
                               random_seed=SEED, verbose=0),
    X_t1, y_dev, fold_indices,
    n_folds=(5 if FULL_CAT else 1),
    fit_kwargs_fn=lambda X_va, y_va: dict(eval_set=(X_va, y_va),
                                          early_stopping_rounds=50))

  [CatBoost] fold 0: Macro-F1=0.1860, Acc=0.1660
  [CatBoost] fold 1: Macro-F1=0.1873, Acc=0.1663
  [CatBoost] fold 2: Macro-F1=0.1849, Acc=0.1645
  [CatBoost] fold 3: Macro-F1=0.1856, Acc=0.1654
  [CatBoost] fold 4: Macro-F1=0.1848, Acc=0.1651
  [CatBoost] 평균 Macro-F1 = 0.1857 ± 0.0010
